In [12]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("Food_Delivery_Times.csv")

# Check dimensions, sample rows, types, and missing counts
print("Shape:", df.shape)
display(df.head())

print("\nColumn info:")
print(df.info())

print("\nMissing values:")
print(df.isna().sum())


Shape: (1000, 9)


,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68



Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order_ID                1000 non-null   int64  
 1   Distance_km             1000 non-null   float64
 2   Weather                 970 non-null    object 
 3   Traffic_Level           970 non-null    object 
 4   Time_of_Day             970 non-null    object 
 5   Vehicle_Type            1000 non-null   object 
 6   Preparation_Time_min    1000 non-null   int64  
 7   Courier_Experience_yrs  970 non-null    float64
 8   Delivery_Time_min       1000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 70.4+ KB
None

Missing values:
Order_ID                   0
Distance_km                0
Weather                   30
Traffic_Level             30
Time_of_Day               30
Vehicle_Type               0
Preparation_Time_min       0
Courier_Ex

In [13]:
from sklearn.model_selection import train_test_split

# Drop row identifier and keep valid positive delivery times
df_clean = df.drop(columns=["Order_ID"]).copy()
df_clean = df_clean.dropna(subset=["Delivery_Time_min"]).copy()
df_clean = df_clean[
    (df_clean["Delivery_Time_min"] > 0) & (df_clean["Distance_km"] > 0)
].copy()

X = df_clean.drop(columns=["Delivery_Time_min"])
y = df_clean["Delivery_Time_min"]

# Split train/test before fitting imputers or encoders
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")

Train rows: 800, Test rows: 200


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_prep, num_cols),
    ("cat", categorical_prep, cat_cols),
])

baseline = Pipeline([
    ("prep", preprocess),
    ("model", DummyRegressor(strategy="median"))
])

ridge = Pipeline([
    ("prep", preprocess),
    ("model", Ridge(alpha=1.0))
])

rf = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])

candidates = {
    "Baseline (Median)": baseline,
    "Ridge Regression": ridge,
    "Random Forest": rf,
}

for label, pipe in candidates.items():
    cv_mae = -cross_val_score(
        pipe, X_train, y_train, cv=5, scoring="neg_mean_absolute_error"
    ).mean()
    print(f"{label} - 5-fold CV MAE: {cv_mae:.2f} min")

# Select model with lowest cross-validation MAE and fit on full training set
best_model = ridge
best_model.fit(X_train, y_train)
baseline.fit(X_train, y_train)

Baseline (Median) - 5-fold CV MAE: 17.88 min
Ridge Regression - 5-fold CV MAE: 6.78 min
Random Forest - 5-fold CV MAE: 7.98 min


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Distance_km',
                                                   'Preparation_Time_min',
                                                   'Courier_Experience_yrs']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Weather', 'Traffic_Level',
                                                   'Time_of_Day',
                                                   'Vehicle_Type'])])),
                ('model', DummyRegressor(strategy='median'))])

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

base_pred = baseline.predict(X_test)
test_pred = best_model.predict(X_test)

base_mae = mean_absolute_error(y_test, base_pred)
test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print(f"Baseline Test MAE: {base_mae:.2f} min")
print(f"Selected Model (Ridge) Test MAE: {test_mae:.2f} min")
print(f"Selected Model (Ridge) Test RMSE: {test_rmse:.2f} min")

# Review largest errors on the test set
errors = X_test.copy()
errors["actual_min"] = y_test.values
errors["pred_min"] = np.round(test_pred, 1)
errors["abs_error_min"] = np.round(
    np.abs(errors["actual_min"] - errors["pred_min"]), 1
)

print("\nLargest test errors:")
display(errors.sort_values("abs_error_min", ascending=False).head(5))

Baseline Test MAE: 17.46 min
Selected Model (Ridge) Test MAE: 6.07 min
Selected Model (Ridge) Test RMSE: 8.95 min

Largest test errors:


,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,actual_min,pred_min,abs_error_min
859,2.99,Foggy,High,Afternoon,Bike,9,5.0,90,37.3,52.7
570,5.93,Foggy,High,NaN,Scooter,27,1.0,106,65.3,40.7
924,17.81,Windy,High,Evening,Bike,21,4.0,122,91.0,31.0
318,4.43,Foggy,Low,Morning,Scooter,10,1.0,63,33.6,29.4
260,16.38,Rainy,Medium,Night,Bike,7,NaN,90,67.6,22.4
